# GP4 ReAct-IR Qwen2.5 QLoRA Kaggle Workflow

Kaggle runtime must persist generated data, reports, checkpoints, adapters, and inference outputs to a Kaggle-backed output or dataset path only.

The notebook uses `gp4_finetune_factory_source_bundle.zip` from `$CLOUD_ROOT/bundles/` when present. Otherwise it clones the pushed `codex/gp4-react-ir-cloud-workflow` branch into the ephemeral runtime. Generated data, reports, checkpoints, adapters, inference outputs, and packages stay under `CLOUD_ROOT`.


In [ ]:
import os
import shutil
import subprocess
import zipfile
from datetime import datetime
from pathlib import Path

RUN_ID = os.environ.get('RUN_ID') or 'gp4-react-v2-300k-' + datetime.utcnow().strftime('%Y%m%d-%H%M')
SOURCE_BRANCH = os.environ.get('GP4_SOURCE_BRANCH', 'codex/gp4-react-ir-cloud-workflow')
GP4_WS_REPO_URL = os.environ.get('GP4_WS_REPO_URL', 'https://github.com/Hieu-RMX18/gp4_ws.git')
GP4_WS_BRANCH = os.environ.get('GP4_WS_BRANCH', 'ws-deep-rebuild-3526')
CLOUD_ROOT = os.environ.get('GP4_CLOUD_ROOT', f'/kaggle/working/gp4_finetune_factory/{RUN_ID}')
assert CLOUD_ROOT.startswith('/kaggle/'), 'Configure GP4_CLOUD_ROOT to a Kaggle-backed storage path.'
SOURCE_BUNDLE = f'{CLOUD_ROOT}/bundles/gp4_finetune_factory_source_bundle.zip'
WORK_DIR = Path('/kaggle/working/gp4_finetune_factory_source')
os.makedirs(f'{CLOUD_ROOT}/reports', exist_ok=True)
os.makedirs(f'{CLOUD_ROOT}/manifests', exist_ok=True)
os.chdir('/kaggle/working')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
if Path(SOURCE_BUNDLE).exists():
    with zipfile.ZipFile(SOURCE_BUNDLE) as archive:
        archive.extractall(WORK_DIR)
else:
    !git clone --branch {SOURCE_BRANCH} --depth 1 https://github.com/Hieu-RMX18/gp4_finetune_factory.git {WORK_DIR}
os.chdir(WORK_DIR)
SEED_PATH = str(WORK_DIR / 'data/seed/gp4_seed_starter.jsonl')
SOURCE_PLAN = str(WORK_DIR / 'docs/superpowers/plans/2026-05-20-gp4-v2-300k-readiness.md')
os.environ['RUN_ID'] = RUN_ID
os.environ['CLOUD_ROOT'] = CLOUD_ROOT
os.environ['SEED_PATH'] = SEED_PATH
os.environ['SOURCE_PLAN'] = SOURCE_PLAN
GP4_WS_EXPECTED_COMMIT = os.environ.get('GP4_WS_EXPECTED_COMMIT', '').strip()
if not GP4_WS_EXPECTED_COMMIT:
    raise RuntimeError('GP4_WS_EXPECTED_COMMIT is required before cloning or reusing the gp4_ws contract snapshot')
if len(GP4_WS_EXPECTED_COMMIT) < 12:
    raise RuntimeError('GP4_WS_EXPECTED_COMMIT must be a full SHA or at least 12 hex characters')
GP4_WS = os.environ.get('GP4_WS', '').strip()
if not GP4_WS:
    GP4_WS = f'{CLOUD_ROOT}/contract_snapshots/gp4_ws_{GP4_WS_BRANCH}'
    if not Path(GP4_WS).exists():
        Path(GP4_WS).parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--branch', GP4_WS_BRANCH, '--depth', '1', GP4_WS_REPO_URL, GP4_WS], check=True)
os.environ['GP4_WS'] = GP4_WS
ACTUAL_GP4_WS_COMMIT = subprocess.run(['git', '-C', GP4_WS, 'rev-parse', 'HEAD'], text=True, capture_output=True, check=True).stdout.strip()
if not ACTUAL_GP4_WS_COMMIT.lower().startswith(GP4_WS_EXPECTED_COMMIT.lower()):
    raise RuntimeError(f'GP4_WS commit mismatch: expected {GP4_WS_EXPECTED_COMMIT}, got {ACTUAL_GP4_WS_COMMIT}')
os.environ['GP4_WS_EXPECTED_COMMIT'] = GP4_WS_EXPECTED_COMMIT
print(CLOUD_ROOT)
print('GP4_WS contract snapshot:', os.environ['GP4_WS'])
print('GP4_WS expected commit:', os.environ['GP4_WS_EXPECTED_COMMIT'])
print('GP4_WS actual commit:', ACTUAL_GP4_WS_COMMIT[:12])
print(WORK_DIR)


In [ ]:
import os
os.environ['DEEPSEEK_BASE_URL'] = os.environ.get('DEEPSEEK_BASE_URL', 'https://api.deepseek.com')
os.environ['OPENAI_MODEL'] = os.environ.get('OPENAI_MODEL', 'gpt-5.4')
os.environ['HF_HOME'] = f'{CLOUD_ROOT}/.cache/huggingface'
os.environ['TRANSFORMERS_CACHE'] = f'{CLOUD_ROOT}/.cache/huggingface/transformers'
os.environ['HF_DATASETS_CACHE'] = f'{CLOUD_ROOT}/.cache/huggingface/datasets'
os.environ['TORCH_HOME'] = f'{CLOUD_ROOT}/.cache/torch'
os.environ['XDG_CACHE_HOME'] = f'{CLOUD_ROOT}/.cache'
os.environ['WANDB_DIR'] = f'{CLOUD_ROOT}/wandb'
os.environ['TMPDIR'] = f'{CLOUD_ROOT}/tmp'
for key in ('HF_HOME', 'TRANSFORMERS_CACHE', 'HF_DATASETS_CACHE', 'TORCH_HOME', 'XDG_CACHE_HOME', 'WANDB_DIR', 'TMPDIR'):
    os.makedirs(os.environ[key], exist_ok=True)


In [ ]:
!python -m pip install -q -r requirements-cloud.txt


In [ ]:
!python scripts/provider_probe.py --provider kaggle --cloud-root "$CLOUD_ROOT" --report "$CLOUD_ROOT/reports/platform_status_${RUN_ID}.json"


In [ ]:
!python scripts/cloud_orchestrator.py --run-id "$RUN_ID" --cloud-root "$CLOUD_ROOT" --seed "$SEED_PATH" --source-plan "$SOURCE_PLAN" --phases ignored --preset v2-300k


In [ ]:
!python scripts/audit_cloud_completion.py --cloud-root "$CLOUD_ROOT" --run-id "$RUN_ID" --report "$CLOUD_ROOT/reports/completion_audit_${RUN_ID}.json"


The orchestrator stops automatically unless the previous gate report has `passed=true`. Completion evidence is `$CLOUD_ROOT/reports/acceptance_gate_report_$RUN_ID.json` with `passed=true`, `$CLOUD_ROOT/reports/benchmark_report_${RUN_ID}.html`, `$CLOUD_ROOT/reports/benchmark_report_${RUN_ID}.md` with Maintenance Reference, and package metadata under the same cloud root.
